# End-to-End: predykcja struktury białek (AlphaFold DB)

Notebook realizuje potok dla zbioru **AlphaFold DB (AF)** (pliki `.cif` w `structures/{train,eval,test}/af/`): z sekwencji aminokwasowych budujemy wejście modelu, przewidujemy współrzędne atomów $C_\alpha$ i zapisujemy wynik w formacie PDB. Potok działa z atrapą (`DummyStructurePredictor`) oraz z dodatkowym, uczonym modelem (`TrainableStructurePredictor`), na którym pokazana jest trenowalność całego szkieletu.

### Architektura potoku:
1. **Sekwencje z plików `.cif` do multi-protein FASTA** i wczytanie ich za pomocą `scikit-bio` (`skbio.io.read()`).
2. **Słownik i kodowanie sekwencji do tensora `(B, D, N)`:**
   * $B$ – rozmiar batcha (np. `16`)
   * $D$ – maksymalna długość sekwencji (`1024`; krótsze sekwencje są dopełniane paddingiem, dłuższe ucinane)
   * $N = 22$ – rozmiar słownika aminokwasów (20 standardowych aminokwasów + `X` dla nieznanych/niestandardowych + `PAD` dla paddingu)
3. **Maskowanie `(B, D)`:** tensor logiczny rozróżniający rzeczywiste reszty (`True`) od paddingu (`False`).
4. **Model PyTorch** przyjmujący zakodowany batch `(B, D, N)` oraz maskę `(B, D)` i zwracający dla każdego białka współrzędne $C_\alpha$ o kształcie **`(L, 3)`**, gdzie $L = \min(\text{długość sekwencji}, D)$. Atrapa ma parametry skali całości i lokalnej geometrii łańcucha.
5. **Zapis wyjścia w formacie wwPDB v3.3:** rekordy `ATOM` dla $C_\alpha$ (1 rekord na przetworzoną resztę, 0 rekordów dla pozycji paddingu).
6. **Dowód trenowalności:** dodatkowy model o tym samym interfejsie, uczony na prawdziwych współrzędnych z plików `.cif` funkcją straty opartą o macierze odległości (dRMSD).

In [1]:
import sys
from pathlib import Path
from typing import List, Tuple, Optional, Union

import numpy as np
import torch
import torch.nn as nn
import skbio.io

# Ustawienie katalogu głównego projektu (aby działało zarówno z folderu projekt/, jak i projekt/notebooks/)
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if (ROOT_DIR / "projekt").exists():
    ROOT_DIR = ROOT_DIR / "projekt"
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f"Katalog projektu: {ROOT_DIR}")
print(f"PyTorch:          {torch.__version__}")
print(f"scikit-bio:       {skbio.__version__}")


Katalog projektu: d:\GitHub Repos\protein-structure-prediction
PyTorch:          2.15.0+cpu
scikit-bio:       0.7.4


---
## Krok 1: Słownik aminokwasów ($N = 22$) i mapowanie na kody 3-literowe

Definiujemy słownik obejmujący:
* **20 standardowych aminokwasów** (indeksy `0..19`),
* **`X`** dla nieznanych lub niestandardowych reszt (indeks `20`),
* **`PAD`** dla pozycji dopełnienia / paddingu (indeks `21`).


In [2]:
CANONICAL_AMINO_ACIDS = [
    'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L',
    'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y'
]

VOCAB = CANONICAL_AMINO_ACIDS + ['X', 'PAD']
AA_TO_IDX = {aa: i for i, aa in enumerate(VOCAB)}
IDX_TO_AA = {i: aa for aa, i in AA_TO_IDX.items()}

VOCAB_SIZE = len(VOCAB)     # N = 22
UNK_IDX = AA_TO_IDX['X']    # 20
PAD_IDX = AA_TO_IDX['PAD']  # 21

# Mapowanie 1-literowe -> 3-literowe wymagane w kolumnach 18-20 formatu PDB
ONE_TO_THREE = {
    'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU', 'F': 'PHE',
    'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'K': 'LYS', 'L': 'LEU',
    'M': 'MET', 'N': 'ASN', 'P': 'PRO', 'Q': 'GLN', 'R': 'ARG',
    'S': 'SER', 'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
    'X': 'UNK', 'PAD': 'PAD'
}

print(f"Rozmiar słownika N = {VOCAB_SIZE}")
print(f"Słownik: {AA_TO_IDX}")


Rozmiar słownika N = 22
Słownik: {'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20, 'PAD': 21}


---
## Krok 2: Przygotowanie i wczytanie pliku multi-protein FASTA z naszych struktur AlphaFold DB (`skbio.io.read()`)

Z naszych pobranych plików AlphaFold DB (`structures/*/af/AF-*.cif`) wyciągnęliśmy sekwencje do pliku `data/samples/sample_multi.fasta`, dobierając białka o różnych długościach, aby przetestować każdy przypadek (padding, ucinanie $>1024$ oraz symbol `X`), a następnie wczytujemy je przez wymagane `skbio.io.read()`:


In [3]:
def parse_fasta_skbio(fasta_path: Union[str, Path]) -> List[Tuple[str, str, str]]:
    """Wczytuje plik multi-protein FASTA przy użyciu skbio.io.read()."""
    fasta_path = Path(fasta_path)
    if not fasta_path.exists():
        raise FileNotFoundError(f"Nie znaleziono pliku FASTA: {fasta_path}")

    records = []
    for seq_obj in skbio.io.read(str(fasta_path), format="fasta"):
        meta = getattr(seq_obj, "metadata", {}) or {}
        pid = meta.get("id", "PROT")
        desc = meta.get("description", "")
        raw_seq = str(seq_obj).strip().upper()
        records.append((pid, desc, raw_seq))
    return records


fasta_path = ROOT_DIR / "data/samples/sample_multi.fasta"
records = parse_fasta_skbio(fasta_path)

print(f"Wczytano {len(records)} białek z pliku: {fasta_path.name}\n")
for pid, desc, seq in records:
    print(f"  • {pid:15s} | długość: {len(seq):4d} AA | {desc}")


Wczytano 4 białek z pliku: sample_multi.fasta

  • AF-A0A075B695-F1-model_v6 | długość:  113 AA | AlphaFold DB structure (113 AA, padding test)
  • AF-A0A077ZFK1-F1-model_v6 | długość:  359 AA | AlphaFold DB structure (359 AA, padding test)
  • AF-A0A1D6IU37-F1-model_v6 | długość: 1112 AA | AlphaFold DB structure (1112 AA > 1024, truncation test)
  • AF_TEST_UNK_F1  | długość:   25 AA | AlphaFold test sequence with non-standard residue X (25 AA)


---
## Krok 3: Kodowanie sekwencji do kształtu `(B, D, N)` oraz tworzenie maski `(B, D)`

* Dla każdej sekwencji obliczamy $L = \min(\text{len(sequence)}, D)$.
* Pozycje `0 .. L-1` są kodowane *one-hot* odpowiednim aminokwasem (lub `X` dla nieznanych symboli), a maska przyjmuje wartość `True`.
* Pozycje `L .. D-1` (padding) otrzymują jedynkę na indeksie `PAD_IDX = 21`, a maska przyjmuje wartość `False`.


In [4]:
def encode_sequence_one_hot(
    sequence: str,
    max_len: int = 1024,
    vocab_size: int = VOCAB_SIZE
) -> Tuple[torch.Tensor, torch.Tensor, int]:
    """Koduje pojedynczą sekwencję do tensora one-hot (D, N) oraz maski (D,)."""
    L = min(len(sequence), max_len)

    one_hot = torch.zeros((max_len, vocab_size), dtype=torch.float32)
    mask = torch.zeros(max_len, dtype=torch.bool)

    # Ważne reszty aminokwasowe (0 .. L-1)
    for i in range(L):
        idx = AA_TO_IDX.get(sequence[i], UNK_IDX)
        one_hot[i, idx] = 1.0
        mask[i] = True

    # Pozycje paddingu (L .. max_len-1)
    if L < max_len:
        one_hot[L:, PAD_IDX] = 1.0

    return one_hot, mask, L


def batch_encode_sequences(
    sequences: List[str],
    max_len: int = 1024
) -> Tuple[torch.Tensor, torch.Tensor, List[int]]:
    """Składa listę sekwencji w batch x o kształcie (B, D, N) oraz maskę (B, D)."""
    batch_x, batch_mask, lengths = [], [], []
    for seq in sequences:
        oh, m, L = encode_sequence_one_hot(seq, max_len=max_len)
        batch_x.append(oh)
        batch_mask.append(m)
        lengths.append(L)
    return torch.stack(batch_x, dim=0), torch.stack(batch_mask, dim=0), lengths


# Test kodowania dla naszych 4 białek z D = 1024
D = 1024
sequences = [r[2] for r in records]
x, mask, lengths = batch_encode_sequences(sequences, max_len=D)

print(f"Kształt tensora wejściowego x : {tuple(x.shape)}  -> (B={len(sequences)}, D={D}, N={VOCAB_SIZE})")
print(f"Kształt maski logicznej mask  : {tuple(mask.shape)}      -> (B={len(sequences)}, D={D})")
print(f"Przetworzone długości L       : {lengths}\n")

for (pid, _, orig_seq), L, m in zip(records, lengths, mask):
    print(f"  • {pid:15s}: orig={len(orig_seq):4d} -> L={L:4d} | ważne (True)={int(m.sum()):4d} | PAD (False)={int((~m).sum()):4d}")


Kształt tensora wejściowego x : (4, 1024, 22)  -> (B=4, D=1024, N=22)
Kształt maski logicznej mask  : (4, 1024)      -> (B=4, D=1024)
Przetworzone długości L       : [113, 359, 1024, 25]

  • AF-A0A075B695-F1-model_v6: orig= 113 -> L= 113 | ważne (True)= 113 | PAD (False)= 911
  • AF-A0A077ZFK1-F1-model_v6: orig= 359 -> L= 359 | ważne (True)= 359 | PAD (False)= 665
  • AF-A0A1D6IU37-F1-model_v6: orig=1112 -> L=1024 | ważne (True)=1024 | PAD (False)=   0
  • AF_TEST_UNK_F1 : orig=  25 -> L=  25 | ważne (True)=  25 | PAD (False)= 999


---
## Krok 4: Model PyTorch atrapy (`DummyStructurePredictor`)

Klasa dziedziczy po `nn.Module`; interfejs (`forward`, `predict_coordinates`) jest wspólny z modelem uczonym z Kroku 8:
* `forward(x, mask=mask)` przyjmuje tensor `(B, D, N)` oraz `mask` `(B, D)` i zwraca tensor `(B, D, 3)`.
* `predict_coordinates(x, mask=mask, lengths=lengths)` odcina padding i dla każdego białka zwraca tensor **`(L, 3)`** ze współrzędnymi $(X, Y, Z)$ atomu $C_\alpha$.

**Parametry atrapy** (wartości domyślne odtwarzają pierwotne zachowanie, czyli niezależne losowe punkty):
* `mode`: `"noise"` (niezależne punkty, brak geometrii lokalnej) lub `"chain"` (łańcuch ze stałą długością wiązania Cα–Cα i zadanym kątem).
* `coord_scale`: skala całości w trybie `"noise"` (odchylenie standardowe każdej współrzędnej, Å).
* `bond_length`, `bond_angle_deg`, `dihedral_range_deg`: lokalna geometria w trybie `"chain"`.
* `proj_weight`: waga niewielkiego wkładu warstwy liniowej zależnej od sekwencji.

Wartości wiarygodne fizycznie opisane są w komentarzach nad parametrami.

In [5]:
import math


class DummyStructurePredictor(nn.Module):
    """
    Atrapa predyktora struktur (bez uczenia). Wejście: x (B, D, N), mask (B, D).
    Wyjście per białko: współrzędne CA o kształcie (L, 3), gdzie L = min(orig_len, D).
    """
    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        max_len: int = 1024,
        coord_dim: int = 3,
        seed: Optional[int] = None,
        # Tryb generowania: "noise" = niezależne losowe punkty (jak w pierwotnej wersji),
        # "chain" = łańcuch o fizycznie wiarygodnej geometrii lokalnej.
        mode: str = "noise",
        # SKALA CAŁOŚCI (tryb "noise"), odchylenie std. każdej współrzędnej w Å.
        # Wiarygodnie: promień bezwładności
        # białka globularnego Rg ≈ 2.2 * L**0.38 Å (przybliżona zależność empiryczna),
        # a std ≈ Rg / sqrt(3): L=100 -> ~7 Å, L=300 -> ~11 Å, L=1000 -> ~17 Å.
        # Wartość 15.0 odpowiada białku o ok. 600-700 resztach.
        coord_scale: float = 15.0,
        # Waga wkładu zależnego od sekwencji (warstwa liniowa). 0.01 jak w pierwotnej wersji;
        # przy coord_scale = 15 jest to wkład pomijalny.
        proj_weight: float = 0.01,
        # LOKALNA GEOMETRIA (tryb "chain"):
        # odległość sąsiednich Cα, Å. Wiarygodnie: 3.8 (typowo 3.80 +/- 0.05; ok. 2.9 dla cis-proliny).
        bond_length: float = 3.8,
        # kąt wirtualny Cα(i-1)-Cα(i)-Cα(i+1), stopnie. Wiarygodnie: ok. 85-130;
        # helisa alfa ok. 91, nić beta ok. 120-125, średnio ok. 105-110.
        bond_angle_deg: float = 110.0,
        # zakres losowania kąta torsyjnego Cα, stopnie (losowo z [-zakres, +zakres]).
        # 180 = pełny zakres (najbardziej swobodny łańcuch). Helisa alfa: ok. +50, nić beta: ok. -170/+190.
        dihedral_range_deg: float = 180.0,
    ):
        super().__init__()
        if mode not in ("noise", "chain"):
            raise ValueError("mode musi być 'noise' lub 'chain'")
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.coord_dim = coord_dim
        self.mode = mode
        self.coord_scale = coord_scale
        self.proj_weight = proj_weight
        self.bond_length = bond_length
        self.bond_angle_deg = bond_angle_deg
        self.dihedral_range_deg = dihedral_range_deg
        self.proj = nn.Linear(vocab_size, coord_dim, bias=True)
        if seed is not None:
            torch.manual_seed(seed)

    @staticmethod
    def _unit(v: torch.Tensor) -> torch.Tensor:
        return v / v.norm(dim=-1, keepdim=True).clamp(min=1e-8)

    def _random_chain(self, B: int, D: int, device, dtype) -> torch.Tensor:
        """Łańcuch Cα o stałej długości wiązania i stałym kącie; kąt torsyjny losowy (metoda NeRF)."""
        b = self.bond_length
        theta = math.radians(self.bond_angle_deg)
        phi_max = math.radians(self.dihedral_range_deg)

        p0 = torch.zeros(B, 3, device=device, dtype=dtype)
        p1 = p0 + torch.tensor([b, 0.0, 0.0], device=device, dtype=dtype)
        p2 = p1 + torch.tensor([-b * math.cos(theta), b * math.sin(theta), 0.0], device=device, dtype=dtype)
        pts = [p0, p1, p2]

        for _ in range(3, D):
            a, bb, c = pts[-3], pts[-2], pts[-1]
            bc = self._unit(c - bb)
            n = self._unit(torch.cross(bb - a, bc, dim=-1))
            m = torch.cross(n, bc, dim=-1)
            phi = (torch.rand(B, 1, device=device, dtype=dtype) * 2 - 1) * phi_max
            d = c + (-b * math.cos(theta)) * bc \
                  + (b * math.sin(theta)) * torch.cos(phi) * m \
                  + (b * math.sin(theta)) * torch.sin(phi) * n
            pts.append(d)
        return torch.stack(pts[:D], dim=1)  # (B, D, 3)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, D, N = x.shape
        if self.mode == "noise":
            base = torch.randn(B, D, self.coord_dim, device=x.device, dtype=x.dtype) * self.coord_scale
        else:
            base = self._random_chain(B, D, x.device, x.dtype)
        coords = base + self.proj_weight * self.proj(x)

        # Zerowanie pozycji paddingu przy użyciu maski
        if mask is not None:
            coords = coords * mask.unsqueeze(-1).to(coords.dtype)
        return coords

    def predict_coordinates(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        lengths: Optional[List[int]] = None
    ) -> List[torch.Tensor]:
        """Zwraca listę tensorów o kształcie (L_i, 3) wyłącznie dla ważnych reszt (bez paddingu)."""
        self.eval()
        with torch.no_grad():
            coords_batch = self.forward(x, mask=mask)  # (B, D, 3)

        results = []
        for b in range(coords_batch.shape[0]):
            if lengths is not None:
                L = lengths[b]
            elif mask is not None:
                L = int(mask[b].sum().item())
            else:
                L = coords_batch.shape[1]
            results.append(coords_batch[b, :L, :].detach().cpu())
        return results


# Inicjalizacja modelu i predykcja współrzędnych (domyślne parametry = pierwotne zachowanie)
model = DummyStructurePredictor(vocab_size=VOCAB_SIZE, max_len=D, seed=42)
predicted_coords = model.predict_coordinates(x, mask=mask, lengths=lengths)

print("Wynik predykcji (kształty wyjściowe per białko):")
for (pid, _, _), coords, L in zip(records, predicted_coords, lengths):
    print(f"  • {pid:15s} -> shape: {tuple(coords.shape)} (oczekiwano: ({L}, 3)) | 1. atom CA: {coords[0].numpy().round(3)}")

Wynik predykcji (kształty wyjściowe per białko):
  • AF-A0A075B695-F1-model_v6 -> shape: (113, 3) (oczekiwano: (113, 3)) | 1. atom CA: [28.903 22.313 13.511]
  • AF-A0A077ZFK1-F1-model_v6 -> shape: (359, 3) (oczekiwano: (359, 3)) | 1. atom CA: [-14.453 -15.811  -9.165]
  • AF-A0A1D6IU37-F1-model_v6 -> shape: (1024, 3) (oczekiwano: (1024, 3)) | 1. atom CA: [-4.617 19.749 11.579]
  • AF_TEST_UNK_F1  -> shape: (25, 3) (oczekiwano: (25, 3)) | 1. atom CA: [-26.53  -10.771  13.275]


---
## Krok 5: Formatowanie i zapis pliku PDB (specyfikacja wwPDB Format v3.3)

Zgodnie z [dokumentacją wwPDB ATOM record](https://www.wwpdb.org/documentation/file-format-content/format33/sect9.html#ATOM) każdy wiersz `ATOM` ma dokładnie **80 znaków** i ściśle określone pozycje kolumn:
* Kolumny `1-6`: `"ATOM  "`
* Kolumny `7-11`: numer seryjny atomu (`%5d`)
* Kolumny `13-16`: nazwa atomu (`" CA "`)
* Kolumny `18-20`: 3-literowa nazwa reszty (`%3s`, np. `ALA`, `UNK` dla `X`)
* Kolumna `22`: identyfikator łańcucha (`A`)
* Kolumny `23-26`: numer sekwencyjny reszty (`%4d`)
* Kolumny `31-38`, `39-46`, `47-54`: współrzędne $X, Y, Z$ w angstremach (`%8.3f`)
* Kolumny `55-60`: occupancy (`1.00`)
* Kolumny `61-66`: czynnik temperaturowy / B-factor (`0.00`)
* Kolumny `77-78`: symbol pierwiastka (`" C"`)

**Ważne:** Zapisujemy dokładnie $L$ rekordów `ATOM` (po jednym na każdą przetworzoną resztę). Pozycje paddingu nie są zapisywane.


In [6]:
def format_ca_atom_record(
    serial: int,
    res_name: str,
    chain_id: str,
    res_seq: int,
    x_coord: float,
    y_coord: float,
    z_coord: float,
    occupancy: float = 1.00,
    temp_factor: float = 0.00
) -> str:
    """Formatuje pojedynczy 80-znakowy rekord ATOM dla atomu CA wg specyfikacji wwPDB v3.3."""
    if len(res_name) == 1:
        res_3 = ONE_TO_THREE.get(res_name.upper(), "UNK")[:3]
    else:
        res_3 = res_name.upper()[:3]
    chain = chain_id[:1] if chain_id else "A"

    return (
        f"ATOM  {serial:5d}  CA  {res_3:3s} {chain:1s}{res_seq:4d}    "
        f"{x_coord:8.3f}{y_coord:8.3f}{z_coord:8.3f}{occupancy:6.2f}{temp_factor:6.2f}          "
        f" C  \n"
    )


def write_ca_pdb(
    coords: Union[torch.Tensor, np.ndarray],
    sequence: str,
    out_path: Union[str, Path],
    protein_id: str = "PROT",
    chain_id: str = "A"
) -> Path:
    """Zapisuje plik PDB zawierający wyłącznie atomy CA dla L przetworzonych reszt (bez paddingu)."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    coords_np = coords.detach().cpu().numpy() if isinstance(coords, torch.Tensor) else np.asarray(coords)
    L = min(len(coords_np), len(sequence))

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"HEADER    PREDICTED STRUCTURE (CA SKELETON) - {protein_id}\n")
        f.write(f"TITLE     PREDICTED CA-ONLY MODEL WITH {L} RESIDUES\n")
        f.write("REMARK    GENERATED BY PSP END-TO-END SKELETON PIPELINE (ALPHAFOLD DB SUBSET)\n")

        for i in range(L):
            x_c, y_c, z_c = coords_np[i]
            f.write(
                format_ca_atom_record(
                    serial=i + 1,
                    res_name=sequence[i],
                    chain_id=chain_id,
                    res_seq=i + 1,
                    x_coord=float(x_c),
                    y_coord=float(y_c),
                    z_coord=float(z_c)
                )
            )

        last_aa = sequence[L - 1] if L > 0 else "X"
        last_res3 = ONE_TO_THREE.get(last_aa, "UNK")[:3]
        f.write(f"TER   {L + 1:5d}      {last_res3:3s} {chain_id:1s}{L:4d}\n")
        f.write("END\n")

    return out_path


# Szybki test formatowania pojedynczego wiersza ATOM:
sample_line = format_ca_atom_record(1, "M", "A", 1, 12.345, -6.789, 0.123).rstrip("\n")
print(f"Przykładowy rekord ATOM ({len(sample_line)} znaków):")
print(sample_line)
print("12345678901234567890123456789012345678901234567890123456789012345678901234567890 (miarka 80 kolumn)")


Przykładowy rekord ATOM (80 znaków):
ATOM      1  CA  MET A   1      12.345  -6.789   0.123  1.00  0.00           C  
12345678901234567890123456789012345678901234567890123456789012345678901234567890 (miarka 80 kolumn)


---
## Krok 6: Pełny potok End-to-End (Multi-Protein FASTA $\to$ Batch `(B, D, N)` $\to$ Pliki PDB)

Łączymy wszystkie elementy w jedną funkcję `run_end_to_end_pipeline`, która:
1. Wczytuje plik FASTA przez `scikit-bio`,
2. Dzieli sekwencje na batche o rozmiarze $B$ (np. `batch_size=16`),
3. Koduje każdy batch do `(B, D, N)` i generuje maskę `(B, D)`,
4. Uruchamia model PyTorch i odbiera współrzędne `(L, 3)` dla każdego białka,
5. Zapisuje gotowe pliki `.pdb` i weryfikuje liczbę rekordów `ATOM`.


In [7]:
out_dir = ROOT_DIR / "predictions/meeting3_pdb"
out_dir.mkdir(parents=True, exist_ok=True)

generated_files = []
for (pid, desc, seq), coords, L in zip(records, predicted_coords, lengths):
    pdb_path = write_ca_pdb(coords=coords, sequence=seq, out_path=out_dir / f"{pid}.pdb", protein_id=pid)
    generated_files.append((pid, len(seq), L, pdb_path))

print(f"Zapisano {len(generated_files)} plików PDB w katalogu: {out_dir}\n")
for pid, orig_len, L, pdb_path in generated_files:
    with open(pdb_path) as f:
        atom_lines = [line for line in f if line.startswith("ATOM  ")]
    assert len(atom_lines) == L, f"Błąd: liczba rekordów ATOM ({len(atom_lines)}) != L ({L})"
    assert all(len(line.rstrip("\r\n")) == 80 for line in atom_lines), "Błąd: wiersz nie ma 80 znaków!"
    print(f"  ✓ {pdb_path.name:20s} | oryg. długość: {orig_len:4d} AA -> zapisano dokładnie {len(atom_lines):4d} rekordów CA ATOM (0 paddingu)")


Zapisano 4 plików PDB w katalogu: d:\GitHub Repos\protein-structure-prediction\predictions\meeting3_pdb

  ✓ AF-A0A075B695-F1-model_v6.pdb | oryg. długość:  113 AA -> zapisano dokładnie  113 rekordów CA ATOM (0 paddingu)
  ✓ AF-A0A077ZFK1-F1-model_v6.pdb | oryg. długość:  359 AA -> zapisano dokładnie  359 rekordów CA ATOM (0 paddingu)
  ✓ AF-A0A1D6IU37-F1-model_v6.pdb | oryg. długość: 1112 AA -> zapisano dokładnie 1024 rekordów CA ATOM (0 paddingu)
  ✓ AF_TEST_UNK_F1.pdb   | oryg. długość:   25 AA -> zapisano dokładnie   25 rekordów CA ATOM (0 paddingu)


### Podgląd wygenerowanego pliku `.pdb` dla białka z AlphaFold DB
Sprawdźmy pierwsze 15 wierszy oraz końcówkę wygenerowanego pliku `AF-A0A075B695-F1-model_v6.pdb` (113 reszt aminokwasowych, 0 rekordów paddingu):


In [8]:
first_pdb = generated_files[0][3]
lines = first_pdb.read_text().splitlines()
print(f"Plik: {first_pdb.name} (łącznie {len(lines)} linii)\n")
print("--- Pierwsze 15 wierszy ---")
print("\n".join(lines[:15]))
print("\n--- Ostatnie 5 wierszy ---")
print("\n".join(lines[-5:]))


Plik: AF-A0A075B695-F1-model_v6.pdb (łącznie 118 linii)

--- Pierwsze 15 wierszy ---
HEADER    PREDICTED STRUCTURE (CA SKELETON) - AF-A0A075B695-F1-model_v6
TITLE     PREDICTED CA-ONLY MODEL WITH 113 RESIDUES
REMARK    GENERATED BY PSP END-TO-END SKELETON PIPELINE (ALPHAFOLD DB SUBSET)
ATOM      1  CA  MET A   1      28.903  22.313  13.511  1.00  0.00           C  
ATOM      2  CA  LEU A   2     -31.581  10.176 -18.519  1.00  0.00           C  
ATOM      3  CA  LEU A   3      -0.644 -24.070 -11.283  1.00  0.00           C  
ATOM      4  CA  ALA A   4      24.729  -5.885 -21.054  1.00  0.00           C  
ATOM      5  CA  LEU A   5     -10.916  -8.392 -11.534  1.00  0.00           C  
ATOM      6  CA  LEU A   6      11.438  24.635  -2.395  1.00  0.00           C  
ATOM      7  CA  PRO A   7      -7.459   6.596 -11.371  1.00  0.00           C  
ATOM      8  CA  VAL A   8      16.177  12.014  25.207  1.00  0.00           C  
ATOM      9  CA  LEU A   9      19.189  19.446   9.156  1.00  0.0

---
## Krok 7: Prawdziwe dane z plików `.cif`

Do uczenia potrzebne są współrzędne referencyjne atomów $C_\alpha$. Poniższy kod:
* czyta plik `.cif` z AlphaFold DB (blok `_atom_site`) i zwraca sekwencję oraz współrzędne `CA`,
* wybiera białka z list `data/splits/<split>_af<SPLIT_SUFFIX>.txt` (funkcje `load_split` i `af_id_to_filename` z `src/data.py`); `SPLIT_SUFFIX = "_30"` oznacza wariant standardowy (4000 / 500 / 500), a `""` wariant duży (40 000 / 5 000 / 5 000),
* zapisuje sekwencje z listy plików `.cif` do multi-protein FASTA (`cifs_to_fasta`),
* liczy statystyki odległości sąsiednich $C_\alpha$ (`ca_bond_stats`; wiarygodna wartość to ok. 3.8 Å).

In [9]:
import shlex
import random

from src.data import load_split, af_id_to_filename

SPLIT_SUFFIX = "_30"   # wariant standardowy (4000 / 500 / 500); "" = wariant duży

THREE_TO_ONE = {v: k for k, v in ONE_TO_THREE.items() if k not in ("X", "PAD")}


def read_cif_ca(cif_path: Union[str, Path]) -> Tuple[str, np.ndarray]:
    """Zwraca (sekwencja, współrzędne CA (L, 3)) z pliku mmCIF (blok _atom_site)."""
    lines = Path(cif_path).read_text(encoding="utf-8").splitlines()
    n, i = len(lines), 0
    while i < n:
        if lines[i].strip() == "loop_":
            j, cols = i + 1, []
            while j < n and lines[j].startswith("_"):
                cols.append(lines[j].strip())
                j += 1
            if cols and cols[0].startswith("_atom_site."):
                ix = {c: k for k, c in enumerate(cols)}
                i_atom, i_res = ix["_atom_site.label_atom_id"], ix["_atom_site.label_comp_id"]
                i_x, i_y, i_z = ix["_atom_site.Cartn_x"], ix["_atom_site.Cartn_y"], ix["_atom_site.Cartn_z"]
                seq, xyz = [], []
                while j < n and not lines[j].startswith(("#", "loop_", "_")):
                    try:
                        f = shlex.split(lines[j])
                    except ValueError:
                        f = lines[j].split()
                    if len(f) >= len(cols) and f[i_atom] == "CA":
                        seq.append(THREE_TO_ONE.get(f[i_res], "X"))
                        xyz.append([float(f[i_x]), float(f[i_y]), float(f[i_z])])
                    j += 1
                return "".join(seq), np.asarray(xyz, dtype=np.float32)
            i = j
        else:
            i += 1
    raise ValueError(f"Brak bloku _atom_site w pliku: {cif_path}")


def cifs_to_fasta(cif_paths: List[Path], out_path: Union[str, Path]) -> Path:
    """Zapisuje sekwencje z listy plików .cif do multi-protein FASTA (do wczytania przez skbio)."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for p in cif_paths:
            seq, _ = read_cif_ca(p)
            f.write(f">{Path(p).stem} AlphaFold DB structure ({len(seq)} AA)\n{seq}\n")
    return out_path


def load_af_split(split: str, max_files: Optional[int] = None, max_len: int = 1024,
                  suffix: str = SPLIT_SUFFIX):
    """Wczytuje (id, sekwencja, coords (L, 3)) dla białek z data/splits/<split>_af<suffix>.txt.
    Pliki spoza listy są ignorowane, brakujące pliki pomijane; L = min(len, max_len)."""
    ids = load_split(split, splits_dir=str(ROOT_DIR / "data" / "splits"), suffix=suffix)
    folder = ROOT_DIR / "structures" / split / "af"
    items, missing = [], 0
    for pid in ids:
        fname = af_id_to_filename(pid)
        path = folder / fname if fname else None
        if path is None or not path.exists():
            missing += 1
            continue
        seq, xyz = read_cif_ca(path)
        L = min(len(seq), max_len)
        items.append((path.stem, seq[:L], xyz[:L]))
        if max_files is not None and len(items) >= max_files:
            break
    if not items:
        raise FileNotFoundError(f"Brak plików z listy {split}_af{suffix}.txt w {folder}")
    print(f"[{split}{suffix}] wczytano {len(items)} białek (pominięto brakujących: {missing})")
    return items


def ca_bond_stats(coords: Union[torch.Tensor, np.ndarray]) -> dict:
    """Statystyki odległości kolejnych atomów CA (wiarygodnie: średnia ok. 3.8 Å, mały rozrzut)."""
    c = coords.detach().cpu().numpy() if isinstance(coords, torch.Tensor) else np.asarray(coords)
    d = np.linalg.norm(c[1:] - c[:-1], axis=1)
    return {"mean": float(d.mean()), "std": float(d.std()), "min": float(d.min()), "max": float(d.max())}

---
## Krok 8: Model uczony (`TrainableStructurePredictor`)

Mały model o tym samym interfejsie co atrapa: `forward(x, mask)` przyjmuje `(B, D, N)` oraz `(B, D)` i zwraca `(B, D, 3)`, a `predict_coordinates` zwraca listę tensorów `(L, 3)`. Budowa: osadzenie reszty (warstwa liniowa `N -> hidden`) + uczone osadzenie pozycji w sekwencji + dwuwarstwowy MLP z głowicą `hidden -> 3`. Pozycje paddingu są zerowane maską. Model jest celowo minimalny, a jego rolą jest przenieść gradient przez cały szkielet (kodowanie, maska, model, strata).

**Funkcja straty (`drmsd_loss`)** porównuje macierze odległości między wszystkimi parami reszt (rzeczywistych, według maski) dla przewidzianych i referencyjnych współrzędnych. Taka strata nie zależy od położenia i orientacji struktury w przestrzeni, więc współrzędne referencyjne można podać w ich oryginalnym układzie.

In [10]:
class TrainableStructurePredictor(nn.Module):
    """Mały uczony model: osadzenie reszty + osadzenie pozycji + MLP -> współrzędne CA (B, D, 3)."""
    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        max_len: int = 1024,
        hidden: int = 128,
        # skala wyjścia (Å): współrzędne z głowicy są mnożone przez tę wartość.
        # Wiarygodnie: rzędu std współrzędnych białka (ok. 7-17 Å dla 100-1000 reszt).
        coord_scale: float = 10.0,
    ):
        super().__init__()
        self.max_len = max_len
        self.coord_scale = coord_scale
        self.embed = nn.Linear(vocab_size, hidden)
        self.pos = nn.Embedding(max_len, hidden)
        self.mlp = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, D, _ = x.shape
        h = self.embed(x) + self.pos(torch.arange(D, device=x.device)).unsqueeze(0)
        coords = self.mlp(h) * self.coord_scale
        if mask is not None:
            coords = coords * mask.unsqueeze(-1).to(coords.dtype)
        return coords

    def predict_coordinates(self, x, mask=None, lengths=None) -> List[torch.Tensor]:
        """Zwraca listę tensorów (L_i, 3) wyłącznie dla ważnych reszt (bez paddingu)."""
        self.eval()
        with torch.no_grad():
            coords_batch = self.forward(x, mask=mask)
        results = []
        for b in range(coords_batch.shape[0]):
            L = lengths[b] if lengths is not None else int(mask[b].sum().item())
            results.append(coords_batch[b, :L, :].detach().cpu())
        return results


def pairwise_distances(c: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Macierz odległości (B, L, L) dla współrzędnych (B, L, 3)."""
    diff = c.unsqueeze(2) - c.unsqueeze(1)
    return torch.sqrt((diff ** 2).sum(dim=-1) + eps)


def drmsd_loss(pred: torch.Tensor, true: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """dRMSD: pierwiastek ze średniego kwadratu różnic odległości par rzeczywistych reszt (bez przekątnej)."""
    L = pred.shape[1]
    pair = (mask.unsqueeze(2) & mask.unsqueeze(1)) & ~torch.eye(L, dtype=torch.bool, device=pred.device)
    pair = pair.to(pred.dtype)
    sq = (pairwise_distances(pred) - pairwise_distances(true)) ** 2 * pair
    per_protein = sq.sum(dim=(1, 2)) / pair.sum(dim=(1, 2)).clamp(min=1)
    return torch.sqrt(per_protein + 1e-8).mean()

---
## Krok 9: Dowód trenowalności

Mały batch prawdziwych białek z listy treningowej przechodzi przez cały szkielet: kodowanie `(B, D, N)` z maską, model, strata dRMSD względem współrzędnych z plików `.cif`. Wykres wartości straty w kolejnych krokach oraz kontrola gradientów pokazują, że wszystkie parametry modelu otrzymują gradient, a strata maleje.

Parametry eksperymentu:
* `PROOF_N` – liczba białek w batchu,
* `PROOF_MAX_LEN` – maksymalna długość wybieranych białek (krótsze białka skracają obliczenia),
* `PROOF_STEPS`, `PROOF_LR` – liczba kroków i współczynnik uczenia optymalizatora Adam.

In [11]:
# --- Parametry dowodu trenowalności ---
PROOF_N = 8            # liczba białek w batchu
PROOF_MAX_LEN = 300    # maksymalna długość wybieranych białek
PROOF_STEPS = 300      # liczba kroków optymalizacji
PROOF_LR = 1e-3        # współczynnik uczenia (Adam)
PROOF_SEED = 42

torch.manual_seed(PROOF_SEED)

pool = load_af_split("train", max_files=200, max_len=D)
proof_items = [it for it in pool if len(it[1]) <= PROOF_MAX_LEN][:PROOF_N]
assert proof_items, "Brak białek spełniających PROOF_MAX_LEN w wczytanej puli"

x_p, mask_p, len_p = batch_encode_sequences([s for _, s, _ in proof_items], max_len=D)
y_p = torch.zeros(len(proof_items), D, 3)
for b, (_, _, c) in enumerate(proof_items):
    y_p[b, :len(c)] = torch.from_numpy(c)
L_crop = max(len_p)   # macierze odległości liczone tylko dla początkowych L_crop pozycji

trainable = TrainableStructurePredictor(vocab_size=VOCAB_SIZE, max_len=D)
optimizer = torch.optim.Adam(trainable.parameters(), lr=PROOF_LR)
n_params = sum(p.numel() for p in trainable.parameters())
print(f"Batch: {len(proof_items)} białek, długości {len_p} | parametry modelu: {n_params:,}")

losses = []
trainable.train()
for step in range(PROOF_STEPS + 1):
    pred = trainable(x_p, mask=mask_p)
    loss = drmsd_loss(pred[:, :L_crop], y_p[:, :L_crop], mask_p[:, :L_crop])
    optimizer.zero_grad()
    loss.backward()

    if step == 0:
        grads = {n: p.grad for n, p in trainable.named_parameters()}
        assert all(g is not None and torch.isfinite(g).all() for g in grads.values()), "Brak gradientu w parametrze"
        total_grad = torch.sqrt(sum((g ** 2).sum() for g in grads.values())).item()
        print(f"Gradient: wszystkie {len(grads)} tensory parametrów mają skończony gradient, norma = {total_grad:.4f}")

    losses.append(loss.item())
    if step % 50 == 0:
        print(f"krok {step:4d} | dRMSD = {loss.item():8.3f} Å")
    if step < PROOF_STEPS:
        optimizer.step()

print(f"\nStrata: {losses[0]:.3f} Å -> {losses[-1]:.3f} Å (spadek o {100 * (1 - losses[-1] / losses[0]):.1f}%)")
assert losses[-1] < losses[0], "Strata nie zmalała"

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig_path = ROOT_DIR / "predictions" / "trainability_loss.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(6, 3.5))
    plt.plot(losses)
    plt.xlabel("krok optymalizacji")
    plt.ylabel("dRMSD [Å]")
    plt.title("Strata podczas uczenia (mały batch)")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.close()
    print(f"Wykres zapisano: {fig_path}")
except ImportError:
    pass

[train_30] wczytano 200 białek (pominięto brakujących: 0)
Batch: 8 białek, długości [121, 114, 116, 117, 102, 117, 97, 121] | parametry modelu: 167,427
Gradient: wszystkie 9 tensory parametrów mają skończony gradient, norma = 3.0219
krok    0 | dRMSD =   27.153 Å
krok   50 | dRMSD =   12.959 Å
krok  100 | dRMSD =    8.802 Å
krok  150 | dRMSD =    6.357 Å
krok  200 | dRMSD =    5.019 Å
krok  250 | dRMSD =    4.524 Å
krok  300 | dRMSD =    4.258 Å

Strata: 27.153 Å -> 4.258 Å (spadek o 84.3%)
Wykres zapisano: d:\GitHub Repos\protein-structure-prediction\predictions\trainability_loss.png


---
## Krok 10: Przepuszczenie danych przez model uczony i zapis PDB

Te same 4 białka z `sample_multi.fasta` (batch `x`, maska `mask` i długości `lengths` z Kroku 3) przechodzą przez uczony model. Wyjście `(L, 3)` zapisywane jest do plików PDB w `predictions/trainable_pdb` tą samą funkcją `write_ca_pdb` co w Kroku 6, a liczba rekordów `ATOM` jest kontrolowana dla każdego pliku.

In [12]:
trainable_dir = ROOT_DIR / "predictions/trainable_pdb"
trainable_dir.mkdir(parents=True, exist_ok=True)

trainable_coords = trainable.predict_coordinates(x, mask=mask, lengths=lengths)

print(f"Zapisano pliki PDB w katalogu: {trainable_dir}\n")
for (pid, desc, seq), coords, L in zip(records, trainable_coords, lengths):
    pdb_path = write_ca_pdb(coords=coords, sequence=seq, out_path=trainable_dir / f"{pid}.pdb", protein_id=pid)
    with open(pdb_path) as f:
        atom_lines = [line for line in f if line.startswith("ATOM  ")]
    assert len(atom_lines) == L, f"Błąd: liczba rekordów ATOM ({len(atom_lines)}) != L ({L})"
    assert all(len(line.rstrip("\r\n")) == 80 for line in atom_lines), "Błąd: wiersz nie ma 80 znaków!"
    print(f"  ✓ {pdb_path.name:32s} | oryg. długość: {len(seq):4d} AA -> zapisano {len(atom_lines):4d} rekordów CA ATOM")

Zapisano pliki PDB w katalogu: d:\GitHub Repos\protein-structure-prediction\predictions\trainable_pdb

  ✓ AF-A0A075B695-F1-model_v6.pdb    | oryg. długość:  113 AA -> zapisano  113 rekordów CA ATOM
  ✓ AF-A0A077ZFK1-F1-model_v6.pdb    | oryg. długość:  359 AA -> zapisano  359 rekordów CA ATOM
  ✓ AF-A0A1D6IU37-F1-model_v6.pdb    | oryg. długość: 1112 AA -> zapisano 1024 rekordów CA ATOM
  ✓ AF_TEST_UNK_F1.pdb               | oryg. długość:   25 AA -> zapisano   25 rekordów CA ATOM
